# Ephemeral RayJob — Ray Data

Topic 2 follows the [ephemeral cluster workflow](https://developers.redhat.com/articles/2025/12/03/tame-ray-workloads-openshift-ai-kuberay-and-kueue#the_ephemeral_cluster__self_service__automated_jobs):
`RayJob` + `ManagedClusterConfig` via the CodeFlare SDK.

Runs `scale_data.py` on a KubeRay cluster; the cluster is deleted when the job finishes.

Official docs: [Running Ray workloads from Jupyter](https://docs.redhat.com/en/documentation/red_hat_openshift_ai_self-managed/3.4/html/working_with_distributed_workloads/running-ray-based-distributed-workloads_distributed-workloads)

In [ ]:
# JupyterLab cwd is usually extras/notebooks — find the repo for working_dir.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from workshop_bootstrap import setup_workshop_path

REPO_ROOT = setup_workshop_path()
print(f"Repo root: {REPO_ROOT}")

## Authenticate

Get **server** and **token** from the OpenShift Console (not from `oc whoami` inside the workbench):

1. OpenShift Console → your username → **Copy login command** → Display token
2. Paste values below

Docs: [Using the cluster server and token to authenticate](https://docs.redhat.com/en/documentation/red_hat_openshift_ai_self-managed/3.4/html/working_with_distributed_workloads/preparing-the-distributed-training-environment_distributed-workloads#using-the-cluster-server-and-token-to-authenticate_preparing-the-distributed-training-environment)

In [ ]:
from codeflare_sdk import TokenAuthentication, list_local_queues

OPENSHIFT_SERVER = "https://api.YOUR_CLUSTER:6443"
OPENSHIFT_TOKEN = "REPLACE_WITH_YOUR_TOKEN"

# skip_tls=True for lab clusters with self-signed certs (article default is False).
auth = TokenAuthentication(
    token=OPENSHIFT_TOKEN,
    server=OPENSHIFT_SERVER,
    skip_tls=True,
)
auth.login()

NAMESPACE = "ray-workshop"
LOCAL_QUEUE = "ray-workshop-queue"

list_local_queues(NAMESPACE)

## Submit ephemeral RayJob

Same pattern as the article: define resources inline with `ManagedClusterConfig`, package the repo as `working_dir`, submit and forget.

In [ ]:
from codeflare_sdk import RayJob, ManagedClusterConfig

job = RayJob(
    job_name="ray-workshop-scale-data",
    entrypoint="python extras/scripts/scale_data.py",
    namespace=NAMESPACE,
    local_queue=LOCAL_QUEUE,
    cluster_config=ManagedClusterConfig(
        num_workers=2,
        head_cpu_requests=1,
        head_cpu_limits=2,
        head_memory_requests=2,
        head_memory_limits=4,
        worker_cpu_requests=1,
        worker_cpu_limits=2,
        worker_memory_requests=2,
        worker_memory_limits=4,
    ),
    runtime_env={
        "working_dir": str(REPO_ROOT),
        "pip": ["pyarrow>=14", "pandas>=2"],
        "env_vars": {
            "INPUT_PATH": "extras/data/iris.csv",
            "OUTPUT_DIR": "/tmp/ray-workshop-output/iris",
        },
    },
    ttl_seconds_after_finished=600,
)

job.submit()
print(f"Submitted: {job.job_name}")

In [ ]:
import time

terminal = {"SUCCEEDED", "FAILED", "STOPPED", "STOPPING"}
deadline = time.time() + 900

while time.time() < deadline:
    status = str(job.status())
    print(f"RayJob {job.job_name}: {status}")
    if status.upper() in terminal:
        break
    time.sleep(15)
else:
    raise TimeoutError(f"Timed out waiting for RayJob {job.job_name}")

print(job.logs())

## Verify

1. OpenShift AI → Distributed workloads (or search RayJobs in the console).
2. Confirm `ray-workshop-scale-data` succeeded.
3. Optional CLI:

```sh
oc get rayjob -n ray-workshop
oc logs -n ray-workshop -l ray.io/node-type=head -c ray-head --tail=80
```

Look for `Done. Wrote N parquet file(s)` in the logs.